<a href="https://colab.research.google.com/github/manaldh14dh-pixel/mfm_aidetect/blob/main/PART1mfmaidetect_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

####ONLY USE T4 GPU

In [ ]:
!pip install transformers datasets stanza openpyxl nltk pandas tqdm

In [ ]:
import pandas as pd
from datasets import load_dataset
from pathlib import Path
import IPython.display as display
from collections import Counter
import re
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
import nltk
import stanza
from tqdm import tqdm
import os
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoModelForCausalLM
import torch
import math

# Creating a folder for processed data so the code doesn't crash later
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

print("ready")

In [ ]:
# تحميل الداتا من Hugging Face
dataset = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts")

print("keys:")
print(dataset.keys())

# تقسيم الداتا
by_polishing = dataset["by_polishing"]
from_title = dataset["from_title"]
from_title_and_content = dataset["from_title_and_content"]

# تحويل لبيانات بايثون
by_polishing_df = by_polishing.to_pandas()
from_title_df = from_title.to_pandas()
from_title_and_content_df = from_title_and_content.to_pandas()
dataset_df = pd.concat([by_polishing_df, from_title_df, from_title_and_content_df], ignore_index=True)

# Save CSVs (As per original code logic)
by_polishing_df.to_csv("by_polishing.csv", index=False, encoding="utf-8-sig")
from_title_df.to_csv("from_title.csv", index=False, encoding="utf-8-sig")
from_title_and_content_df.to_csv("from_title_and_content.csv", index=False, encoding="utf-8-sig")
dataset_df.to_csv("dataset.csv", index=False, encoding="utf-8-sig")
print("تم حفظ البيانات في CSV")

# Re-read CSVs (Original Logic)
by_polishing = pd.read_csv('by_polishing.csv', encoding="utf-8-sig")
from_title = pd.read_csv('from_title.csv', encoding="utf-8-sig")
from_title_and_content = pd.read_csv('from_title_and_content.csv', encoding="utf-8-sig")
dataset = pd.read_csv('dataset.csv', encoding="utf-8-sig")

In [ ]:
# إنشاء داتا مصنفة (Human vs AI)

#by polishing method
human_df_by_polishing = pd.DataFrame({
    'text': by_polishing['original_abstract'],
    'label': 'human',
    'genration_method':'by_polishing'
})

ai_df_by_polishing = pd.DataFrame({
    'text': pd.concat([
        by_polishing['allam_generated_abstract'],
        by_polishing['jais_generated_abstract'],
        by_polishing['llama_generated_abstract'],
        by_polishing['openai_generated_abstract']
    ]),
    'label': 'ai',
    'genration_method':'by_polishing'
})

#from title method
human_df_from_title = pd.DataFrame({
    'text': from_title['original_abstract'],
    'label': 'human',
    'genration_method':'from_title'
})

ai_df_from_title = pd.DataFrame({
    'text': pd.concat([
        from_title['allam_generated_abstract'],
        from_title['jais_generated_abstract'],
        from_title['llama_generated_abstract'],
        from_title['openai_generated_abstract']
    ]),
    'label': 'ai',
     'genration_method':'from_title'
})

#from_title_and_content method
human_df_from_title_and_content = pd.DataFrame({
    'text': from_title_and_content['original_abstract'],
    'label': 'human',
    'genration_method':'from_title_and_content'
})

ai_df_from_title_and_content = pd.DataFrame({
    'text': pd.concat([
        from_title_and_content['allam_generated_abstract'],
        from_title_and_content['jais_generated_abstract'],
        from_title_and_content['llama_generated_abstract'],
        from_title_and_content['openai_generated_abstract']
    ]),
    'label': 'ai',
    'genration_method':'from_title_and_content'
})

# Dataframe merge
merged_df_dataset = pd.concat([human_df_by_polishing, ai_df_by_polishing,human_df_from_title, ai_df_from_title,human_df_from_title_and_content, ai_df_from_title_and_content], ignore_index=True)
merged_df_by_polishing = pd.concat([human_df_by_polishing, ai_df_by_polishing], ignore_index=True)
merged_df_from_title = pd.concat([human_df_from_title, ai_df_from_title], ignore_index=True)
merged_df_from_title_and_content = pd.concat([human_df_from_title_and_content, ai_df_from_title_and_content], ignore_index=True)

# Remove duplicates
merged_df_dataset = merged_df_dataset.drop_duplicates(ignore_index=True)

# Save to Excel (Original Logic)
with pd.ExcelWriter("data/raw/all_datasets.xlsx", engine="openpyxl") as writer:
    merged_df_by_polishing.to_excel(writer, sheet_name="by_polishing", index=False)
    merged_df_from_title.to_excel(writer, sheet_name="from_title", index=False)
    merged_df_from_title_and_content.to_excel(writer, sheet_name="from_title_and_content", index=False)
    merged_df_dataset.to_excel(writer, sheet_name="Merged_Full_Dataset", index=False)

print("Initial Merge Complete and Saved.")

###CHANGE 1

In [ ]:
# ==========================================
# PHASE 1: STRUCTURAL FEATURES & PREPROCESSING
# ==========================================

# Reloading the dataframe from the previous step (just to be safe)
if 'merged_df_dataset' in locals():
    df = merged_df_dataset.copy()
elif 'df' not in locals():
    # Fallback if variable was lost
    df = pd.read_excel("data/raw/all_datasets.xlsx", sheet_name="Merged_Full_Dataset")

print(f"Working with {len(df)} rows.")

# ---------------------------------------------------------
# STEP 1: CALCULATE STRUCTURAL FEATURES (ON RAW TEXT)
# ---------------------------------------------------------
print("1. Calculating Structural Features (Paragraphs, Sentences, Diacritics)...")

def get_structural_features(text):
    if not isinstance(text, str) or not text.strip():
        # Added 0 at the end for num_paragraphs
        return pd.Series([0, 0, 0, 0, 0, 0])

    # 1. Diacritic Ratio (Feature 101)
    diacritics = re.findall(r'[\u064B-\u0652]', text)
    diacritic_ratio = (len(diacritics) / len(text)) * 100

    # 2. Paragraphs (Feature 35/38)
    paragraphs = [p.strip() for p in re.split(r'[\r\n]+', text) if p.strip()]
    num_paragraphs = len(paragraphs) if paragraphs else 1
    chars_per_para = len(text.replace(" ", "")) / num_paragraphs

    # 3. Sentences (Feature 34/39/80)
    sentences = re.split(r'[.!؟\n]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences) if sentences else 1

    words = text.split()
    num_words = len(words)

    # Avg Words per Sentence (Feature 39)
    avg_words_per_sent = num_words / num_sentences

    # Flesch-Kincaid (Feature 80)
    syllables = sum(len(re.findall(r'[aeiouAEIOU\u0627\u0648\u064A]', w)) for w in words)
    if num_words > 0:
        flesch_score = 0.39 * (num_words / num_sentences) + 11.8 * (syllables / num_words) - 15.59
    else:
        flesch_score = 0

    # ADDED: num_paragraphs to return
    return pd.Series([diacritic_ratio, chars_per_para, avg_words_per_sent, flesch_score, num_sentences, num_paragraphs])

# Apply structurally - ADDED 'feat_35_num_paragraphs' to the list
df[['feat_101_diacritic_ratio', 'feat_38_chars_per_para',
    'feat_39_avg_words_per_sentence', 'feat_80_flesch_kincaid',
    'feat_34_num_sentences', 'feat_35_num_paragraphs']] = df['text'].apply(get_structural_features)


# ---------------------------------------------------------
# STEP 2: CLEAN TEXT (DESTRUCTIVE STEP)
# ---------------------------------------------------------
print("2. Cleaning Text (Removing Punctuation & Diacritics)...")

nltk.download('stopwords', quiet=True)
from nltk.stem.isri import ISRIStemmer
stemmer = ISRIStemmer()
arabic_stopwords = set(stopwords.words('arabic'))

def preprocess_text_v2(text):
    if not isinstance(text, str): return ""

    # Normalize Alef
    text = re.sub(r'[إأآ]', 'ا', text)
    text = re.sub(r'ى', 'ا', text)
    text = re.sub(r'ـ', '', text) # Tatweel

    # Remove non-Arabic letters (This removes punctuation!)
    #text = re.sub(r'[^\u0621-\u064A\s]', '', text)

    # Fixed to include letters, spaces, Western digits (0-9), and Eastern digits (٠-٩)
    text = re.sub(r'[^\u0621-\u064A\u0660-\u0669\s0-9]', '', text)

    # Stopwords & Stemming
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in arabic_stopwords]

    return ' '.join(words)

df["processed_text"] = df["text"].apply(preprocess_text_v2)


# ---------------------------------------------------------
# STEP 3: CALCULATE WORD-BASED FEATURES (ON CLEAN TEXT)
# ---------------------------------------------------------
print("3. Calculating Word-Based Features (Yule's K, etc.)...")

# Feature 17: Yule's K
def yules_k_measure(text):
    words = text.split()
    N = len(words)
    if N == 0: return 0
    freqs = Counter(words)
    numerator = sum(f*(f-1) for f in freqs.values())
    return 10000 * numerator / (N*N)

df["feat_17_yules_k"] = df["processed_text"].apply(yules_k_measure)

# Feature 18: Simpson's Diversity
def simpsons_diversity(text):
    words = text.split()
    N = len(words)
    if N == 0: return 0
    freqs = Counter(words)
    numerator = sum(f*(f-1) for f in freqs.values())
    denominator = N * (N - 1)
    return 1 - (numerator / denominator) if denominator > 0 else 0

df["feat_18_simpsons"] = df["processed_text"].apply(simpsons_diversity)

print("\n✓ Phase 1 Complete.")
display.display(df[['text', 'processed_text', 'feat_35_num_paragraphs', 'feat_17_yules_k']].head(3))

In [ ]:
# =========================================
# FINAL REPORT: READ FROM RAW XLSX
# =========================================
import pandas as pd
import re
from collections import Counter

# 1. Read directly from the raw xlsx file
file_path = "/content/Final_Engineered_Dataset.xlsx"
print(f"\nReading data directly from: {file_path} ...")
df_final = pd.read_excel(file_path)

print("\n" + "="*40)
print("     ملخص جودة البيانات (من الملف الخام)")
print("="*40)

# 2. Missing Values (عدد القيم المفقودة)
print("\n:عدد القيم المفقودة")
cols_to_check = ['text', 'label', 'genration_method']
# Filter columns that actually exist in the file
present_cols = [c for c in cols_to_check if c in df_final.columns]
print(df_final[present_cols].isnull().sum())

# 3. Duplicates (عدد الصفوف المكررة)
print("\n:عدد الصفوف المكررة")
print(df_final.duplicated().sum())

# 4. Unique Labels (القيم الفريدة في label)
print("\n:label القيم الفريدة في")
if 'label' in df_final.columns:
    print(df_final['label'].unique())

# 5. Strange Symbols Calculation (using 'text' column)
def get_strange_chars(text):
    # Ensure text is string; Regex finds Non-Arabic, Non-Number, Non-Space
    if not isinstance(text, str): return []
    return re.findall(r"[^0-9\u0600-\u06FF\s]", text)

# Flatten list of all strange characters found
all_strange_tokens = [char for text in df_final['text'] for char in get_strange_chars(text)]
strange_counts = Counter(all_strange_tokens)
unique_symbols = sorted(strange_counts.keys())
total_strange_count = sum(strange_counts.values())

# 6. Strange Symbols Output (الرموز الغريبة الموجودة)
print("\n:الرموز الغريبة الموجودة (بدون تكرار)")
# Join them with commas and wrap in braces to match screenshot style
print(f"{{{', '.join([repr(s) for s in unique_symbols])}}}")

# 7. Total Strange Symbols (مجموع الرموز الغريبة)
print("\n:مجموع الرموز الغريبة")
print(total_strange_count)
print("="*40)

###CHANGE 2

In [ ]:
# ==========================================
# PHASE 2: Deep Learning Models and Stanza CORRECTED
# ==========================================

import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoModelForCausalLM
import gc
from tqdm import tqdm
import math
import os
import pandas as pd

# OPTIONAL: Helps with fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

# ---------------------------------------------------------
# PART A: AraBERT Features (Feat 59, 60 & 98)
# ---------------------------------------------------------
print("\n--- Loading AraBERT (for Features 59, 60 & 98) ---")
bert_model_name = "aubmindlab/bert-base-arabertv02"

try:
    bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
    # LOAD IN HALF PRECISION (float16) to save memory while keeping high batch size
    bert_model = AutoModel.from_pretrained(bert_model_name, dtype=torch.float16).to(device)
    bert_model.eval()
    print("✓ AraBERT Loaded (Float16).")
except Exception as e:
    print(f"Error loading AraBERT: {e}")

# Feat 59 & 60: Unique Embedding Words (500 and 1000)
# BATCH: 1024 (Kept as requested)
def get_embedding_uniqueness_both(text_list, batch_size=1024):
    results_500 = []
    results_1000 = []
    total = len(text_list)
    for i in tqdm(range(0, total, batch_size), desc="Feature 59 & 60 (Embeddings)"):
        batch_texts = text_list[i:i+batch_size]
        # We process 1000 tokens max here to cover both features
        enc = bert_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=1000).to(device)
        input_ids = enc["input_ids"]
        for j in range(len(batch_texts)):
            # Feature 59 (First 500)
            tokens_500 = input_ids[j][:500]
            decoded_500 = bert_tokenizer.decode(tokens_500, skip_special_tokens=True)
            results_500.append(len(set(decoded_500.split())))

            # Feature 60 (First 1000)
            tokens_1000 = input_ids[j][:1000]
            decoded_1000 = bert_tokenizer.decode(tokens_1000, skip_special_tokens=True)
            results_1000.append(len(set(decoded_1000.split())))

    return results_500, results_1000

# Feat 98: CLS Score
# BATCH: 512 (Kept as requested)
def get_cls_score(text_list, batch_size=512):
    results = []
    total = len(text_list)
    # Using autocast for safety, though model is already float16
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for i in tqdm(range(0, total, batch_size), desc="Feature 98 (CLS Score)"):
            batch_texts = text_list[i:i+batch_size]
            enc = bert_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            outputs = bert_model(**enc)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            scores = torch.mean(torch.abs(cls_embeddings), dim=1).cpu().numpy().tolist()
            results.extend(scores)
    return results

# --- FIX: Use RAW TEXT for Deep Learning ---
print("Using RAW text for Deep Learning features...")
texts = df['text'].astype(str).tolist()

# Calculate Feature 59 and 60
feat59, feat60 = get_embedding_uniqueness_both(texts, batch_size=1024)
df['feat_59_unique_embedding_words_500'] = feat59
df['feat_60_unique_embedding_words_1000'] = feat60

# Calculate Feature 98
df['feat_98_bert_cls_score'] = get_cls_score(texts, batch_size=512)

# FREE MEMORY
del bert_model
del bert_tokenizer
torch.cuda.empty_cache()
gc.collect()
print("✓ AraBERT features done & memory cleared.")


# ---------------------------------------------------------
# PART B: AraGPT2 Features (Feat 81 - LogRank)
# ---------------------------------------------------------
print("\n--- Loading AraGPT2 (for Feature 81 LogRank) ---")
gpt_model_name = "aubmindlab/aragpt2-medium"

try:
    gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)
    # LOAD IN HALF PRECISION (float16)
    gpt_model = AutoModelForCausalLM.from_pretrained(gpt_model_name, dtype=torch.float16).to(device)
    gpt_model.eval()
    print("✓ AraGPT2 Loaded (Float16).")
except Exception as e:
    print(f"Error loading AraGPT2: {e}")

# Feat 81: LogRank
def calculate_logrank(text_list, batch_size=8):
    results = []
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

    with torch.no_grad(), torch.amp.autocast('cuda'):
        for i in tqdm(range(0, len(text_list), batch_size), desc="Feature 81 (LogRank)"):
            batch_texts = text_list[i:i+batch_size]
            inputs = gpt_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            outputs = gpt_model(**inputs)

            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = inputs["input_ids"][..., 1:].contiguous()
            shift_mask = inputs["attention_mask"][..., 1:].contiguous()
            log_probs = torch.log_softmax(shift_logits, dim=-1)

            for j in range(len(batch_texts)):
                valid_len = torch.sum(shift_mask[j]).item()
                if valid_len <= 0:
                    results.append(0)
                    continue
                token_log_probs = []
                for k in range(valid_len):
                    token_id = shift_labels[j, k].item()
                    lp = log_probs[j, k, token_id].item()
                    token_log_probs.append(-lp)
                avg_log_rank = sum(token_log_probs) / len(token_log_probs) if token_log_probs else 0
                results.append(avg_log_rank)

            del outputs, logits, log_probs, inputs
    return results

df['feat_81_logrank'] = calculate_logrank(texts, batch_size=8)

# FREE MEMORY
del gpt_model
del gpt_tokenizer
torch.cuda.empty_cache()
gc.collect()
print("✓ AraGPT2 features done & memory cleared.")


# ---------------------------------------------------------
# PART C: Stanza Features (Optimized Stream Mode)
# ---------------------------------------------------------
print("\n--- Running Stanza (POS Tagging) - STREAM MODE ---")

try:
    import stanza
    stanza.download('ar', verbose=False)
    nlp_stanza = stanza.Pipeline(lang='ar', processors='tokenize,pos', verbose=False, use_gpu=True)

    # Replace empty text with "." to keep alignment
    texts_for_stanza = [t if isinstance(t, str) and t.strip() else "." for t in df['text']]
    print(f"Streaming {len(texts_for_stanza)} texts to Stanza...")

    results = []
    for doc in tqdm(nlp_stanza.stream(texts_for_stanza), total=len(texts_for_stanza), desc="Feature 56"):
        adj = 0
        adv = 0
        for sent in doc.sentences:
            for word in sent.words:
                if word.upos == 'ADJ': adj += 1
                elif word.upos == 'ADV': adv += 1
        # Laplace Smoothing
        ratio = (adj + 1) / (adv + 1)
        results.append(ratio)

    df['feat_56_adj_adv_ratio'] = results
    print("✓ Stanza features done.")

except Exception as e:
    print(f"Stanza failed: {e}")
    df['feat_56_adj_adv_ratio'] = 0

In [ ]:
# ==========================================
# PHASE 3: RESTORING MISSING FEATURES (14, 77, 102)
# ==========================================
print("\n--- Phase 3: Calculating Missing Statistical Features ---")

# 1. Feature 14: Hapax Dislegomena
def get_hapax_dislegomena(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    words = text.split()
    counts = Counter(words)
    twice_count = sum(1 for count in counts.values() if count == 2)
    total_words = len(words)
    return twice_count / total_words if total_words > 0 else 0.0

print("Calculating Feature 14...")
df['feat_14_hapax_dislegomena'] = df['processed_text'].apply(get_hapax_dislegomena)


# 2. Feature 102: Detailed Diacritic Type Counts
DIACRITICS = {
    "fatha": "َ",
    "damma": "ُ",
    "kasra": "ِ",
    "sukun": "ْ",
    "shadda": "ّ",
    "tanwin_fath": "ً",
    "tanwin_damm": "ٌ",
    "tanwin_kasr": "ٍ"
}

print("Calculating Feature 102 (Detailed Diacritics)...")
# FIX: Use DIACRITICS dictionary, not the undefined 'diacritic_map'
for name, char in DIACRITICS.items():
    df[f'feat_102_{name}'] = df['text'].astype(str).apply(lambda x: x.count(char))


# 3. Feature 77: Corpus-Level Top 100 Words
print("Calculating Feature 77 (Global Top 100)...")

# Combine all processed text
all_text_combined = ' '.join(df['processed_text'].astype(str).tolist())
all_words = all_text_combined.split()
total_corpus_words = len(all_words)
word_counts_global = Counter(all_words)

# Get top 100 words
most_common_100 = word_counts_global.most_common(100)
top_100_freq_sum = sum(count for word, count in most_common_100)

# 1. Global Ratio
feature77_global_value = top_100_freq_sum / total_corpus_words if total_corpus_words > 0 else 0.0
df['feat_77_global_top100_ratio'] = feature77_global_value

# 2. Top 100 Words String (Comma separated list)
# ADDED: This extracts the actual words so they aren't lost, same as original code
top_100_words_list = [word for word, count in most_common_100]
top_100_str = ", ".join(top_100_words_list)
df['feat_77_top100_string'] = top_100_str # Assigned to the whole column (will repeat)

print("✓ Phase 3 Complete.")


# ---------------------------------------------------------
# FINAL SAVE
# ---------------------------------------------------------
print("\nSaving final dataset...")
df.to_excel("data/processed/Final_Engineered_Dataset.xlsx", index=False)
print("SUCCESS! File saved to: data/processed/Final_Engineered_Dataset.xlsx")

In [ ]:
from IPython.display import display

In [ ]:
# ==========================================
# TASK 2.2: SUPERIOR EDA & REPORT ZIPPING (REFACTORED & FIXED)
# ==========================================

# 1. SETUP & LIBRARIES
print("1. Installing visualization libraries...")
!pip install -q arabic-reshaper python-bidi wordcloud seaborn

# Download a proper Arabic font (Amiri) to avoid empty boxes in WordCloud
!wget -q -O amiri_font.ttf https://github.com/google/fonts/raw/main/ofl/amiri/Amiri-Regular.ttf

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import re
import os
import zipfile
import warnings
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from wordcloud import WordCloud
import arabic_reshaper
from bidi.algorithm import get_display
from nltk.corpus import stopwords
import nltk

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

os.makedirs('reports/figures', exist_ok=True)
nltk.download('stopwords', quiet=True)

# Register Arabic font for Matplotlib
fm.fontManager.addfont('amiri_font.ttf')
plt.rcParams['font.family'] = 'Amiri'
plt.rcParams['figure.dpi'] = 120
sns.set_style("whitegrid")

# Helper for Arabic reshaping
def reshaper(text):
    if not isinstance(text, str): return str(text)
    reshaped_text = arabic_reshaper.reshape(text)
    return get_display(reshaped_text)

# 2. LOAD DATA (User Provided Logic)
file_path = "/content/Final_Engineered_Dataset.xlsx"
print(f"\n2. Loading data from {file_path}...")

if os.path.exists(file_path):
    df_eda = pd.read_excel(file_path)
else:
    # Try CSV fallback
    csv_path = "data/processed/Final_Engineered_Dataset.csv" # Fixed typo from .xslx to .csv
    if os.path.exists(csv_path):
        df_eda = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"CRITICAL: Data file not found at {file_path} or {csv_path}")

print(f"Loaded {len(df_eda)} rows.")

# =========================================================
# PART A: RAW DATA HEALTH CHECK & STRANGE CHARACTERS
# =========================================================
print("\n--- A. Raw Data Health Check ---")

# 1. Strange Character Extraction
def extract_strange_list(text):
    if pd.isna(text): return []
    # Regex: Anything NOT Arabic, NOT Number, NOT Space
    return re.findall(r"[^0-9\u0600-\u06FF\s]", str(text))

print("Extracting strange symbols from raw text...")
all_strange_tokens = [token for text in df_eda['text'] for token in extract_strange_list(text)]

# Count frequencies
strange_counts = Counter(all_strange_tokens)
total_strange_count = sum(strange_counts.values())
unique_strange_count = len(strange_counts)

print(f"Found {unique_strange_count} unique strange characters. Total occurrences: {total_strange_count}")

# 2. Save Strange Characters to CSV
df_strange = pd.DataFrame(strange_counts.most_common(), columns=['Character', 'Count'])
str_csv_path = 'reports/strange_characters.csv'
df_strange.to_csv(str_csv_path, index=False)
print(f"Saved strange characters to '{str_csv_path}'")

# 3. General Stats CSV
print("Generating General Info Report...")
health_data = {
    'Metric': ['Total Rows', 'Human Samples', 'AI Samples', 'Missing Values', 'Duplicates', 'Total Strange Chars'],
    'Value': [
        len(df_eda),
        df_eda[df_eda['label']=='human'].shape[0],
        df_eda[df_eda['label']=='ai'].shape[0],
        df_eda['text'].isnull().sum(),
        df_eda.duplicated(subset=['text']).sum(),
        total_strange_count
    ]
}
pd.DataFrame(health_data).to_csv('reports/raw_data_health_check.csv', index=False)

# =========================================================
# PART B: VISUALIZATION (STRANGE CHARACTERS)
# =========================================================
print("\n--- B. Visualizing Strange Characters ---")

def plot_top_strange(data, top_n, filename, height):
    if data.empty: return

    subset = data.head(top_n)

    plt.figure(figsize=(10, height))
    # Fix: Added hue=Character and legend=False to silence FutureWarning
    sns.barplot(data=subset, y='Character', x='Count', palette='viridis', hue='Character', legend=False)
    plt.title(f"Top {top_n} Strange Characters/Symbols")
    plt.xlabel("Frequency")
    plt.ylabel("Character/Symbol")

    # Add labels
    for i, v in enumerate(subset['Count']):
        plt.text(v, i, f" {v}", va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f'reports/figures/{filename}')
    plt.close()
    print(f"Saved: reports/figures/{filename}")

# Generate Graphs (Added Top 10)
plot_top_strange(df_strange, 10, 'strange_chars_top_10.png', 6)
plot_top_strange(df_strange, 25, 'strange_chars_top_25.png', 8)
plot_top_strange(df_strange, 50, 'strange_chars_top_50.png', 12)
plot_top_strange(df_strange, 100, 'strange_chars_top_100.png', 20)

# =========================================================
# PART C: ADVANCED LINGUISTIC ANALYSIS
# =========================================================
print("\n--- C. Advanced Linguistic Analysis ---")

def calc_text_stats(text):
    if not isinstance(text, str): return pd.Series([0, 0, 0])
    words = text.split()
    if len(words) == 0: return pd.Series([0, 0, 0])
    avg_word_len = sum(len(w) for w in words) / len(words)
    ttr = len(set(words)) / len(words)
    sentences = re.split(r'[.!?؟]', text)
    avg_sent_len = len(words) / len(sentences) if len(sentences) > 0 else len(words)
    return pd.Series([avg_word_len, ttr, avg_sent_len])

df_eda[['avg_word_len', 'ttr', 'avg_sent_len']] = df_eda['processed_text'].apply(calc_text_stats)

stats_summary = df_eda.groupby('label')[['avg_word_len', 'ttr', 'avg_sent_len']].mean()
stats_summary.to_csv('reports/linguistic_stats_summary.csv')
display(stats_summary)

# ---------------------------------------------------------
# PART D: PUNCTUATION (Log Scale)
# ---------------------------------------------------------
print("\n--- D. Punctuation Analysis ---")
def count_punct(text):
    return len(re.findall(r'[.,،؛:!؟?]', str(text)))

df_eda['punct_count'] = df_eda['text'].apply(count_punct)
df_eda['punct_density'] = (df_eda['punct_count'] / df_eda['text'].str.len()) * 1000

plt.figure(figsize=(8, 5))
# Fix: Added hue=label
sns.boxplot(data=df_eda, x='label', y='punct_density', palette='Pastel1', hue='label', legend=False)
plt.yscale('symlog')
plt.title("Punctuation Density (Log Scale)")
plt.savefig('reports/figures/eda_punctuation_log_scale.png')
plt.close()

# ---------------------------------------------------------
# PART E: HEATMAP
# ---------------------------------------------------------
print("\n--- E. Feature Correlations ---")
label_map = {'ai': 0, 'human': 1}
df_eda['label_encoded'] = df_eda['label'].map(label_map)

feat_cols = [c for c in df_eda.columns if c.startswith('feat_') and pd.api.types.is_numeric_dtype(df_eda[c])]
cols_to_corr = ['label_encoded'] + feat_cols

if len(feat_cols) > 0:
    plt.figure(figsize=(12, 10))
    corr_matrix = df_eda[cols_to_corr].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm', linewidths=0.5, center=0)
    plt.title("Feature Correlation Matrix (AI=0, Human=1)")
    plt.savefig('reports/figures/eda_feature_correlation_matrix.png')
    plt.close()

# ---------------------------------------------------------
# PART F: PCA PROJECTION
# ---------------------------------------------------------
print("\n--- F. PCA Projection ---")
if len(feat_cols) > 2:
    X = df_eda[feat_cols].fillna(0)
    y = df_eda['label']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    plt.figure(figsize=(10, 7))
    sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, alpha=0.6, palette='viridis', s=15)
    plt.title(f"PCA Projection\nExplained Variance: {sum(pca.explained_variance_ratio_):.2%}")
    plt.savefig('reports/figures/eda_pca_projection.png')
    plt.close()

# ---------------------------------------------------------
# PART G: DISTINCTIVE WORDS
# ---------------------------------------------------------
print("\n--- G. Distinctive Vocabulary ---")
human_idx = df_eda['label'] == 'human'
ai_idx = df_eda['label'] == 'ai'

if human_idx.sum() > 0 and ai_idx.sum() > 0:
    tfidf = TfidfVectorizer(max_features=5000, stop_words=stopwords.words('arabic'))
    X_tfidf = tfidf.fit_transform(df_eda['processed_text'].astype(str))
    words = np.array(tfidf.get_feature_names_out())

    # Safe slicing
    diff = X_tfidf[human_idx.values].mean(axis=0).A1 - X_tfidf[ai_idx.values].mean(axis=0).A1

    top_human = diff.argsort()[-10:][::-1]
    top_ai = diff.argsort()[:10]

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    h_words = [reshaper(words[i]) for i in top_human]
    a_words = [reshaper(words[i]) for i in top_ai]

    # Fix: Added hue for barplots
    sns.barplot(x=diff[top_human], y=h_words, ax=ax[0], palette="Greens_r", hue=h_words, legend=False)
    ax[0].set_title("Most Distinctive HUMAN Words")

    sns.barplot(x=np.abs(diff[top_ai]), y=a_words, ax=ax[1], palette="Blues_r", hue=a_words, legend=False)
    ax[1].set_title("Most Distinctive AI Words")
    plt.tight_layout()
    plt.savefig('reports/figures/eda_distinctive_vocab.png')
    plt.close()

# ---------------------------------------------------------
# PART H: TOP BIGRAMS
# ---------------------------------------------------------
print("\n--- H. Top Bigrams ---")
def plot_save_ngrams(text_series, label, color_pal):
    if len(text_series) == 0: return
    vec = CountVectorizer(ngram_range=(2, 2), max_features=10)
    try:
        bow = vec.fit_transform(text_series.astype(str))
        sum_words = bow.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
        words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)

        words, freqs = zip(*words_freq)
        words_display = [reshaper(w) for w in words]

        plt.figure(figsize=(8, 5))
        # Fix: Added hue
        sns.barplot(x=list(freqs), y=list(words_display), palette=color_pal, hue=list(words_display), legend=False)
        plt.title(f"Top Bigrams: {label.upper()}")
        plt.tight_layout()
        plt.savefig(f'reports/figures/eda_bigrams_{label}.png')
        plt.close()
    except ValueError: pass

plot_save_ngrams(df_eda[df_eda['label']=='human']['processed_text'], 'human', 'Greens_r')
plot_save_ngrams(df_eda[df_eda['label']=='ai']['processed_text'], 'ai', 'Blues_r')

# ---------------------------------------------------------
# PART I: WORD CLOUDS
# ---------------------------------------------------------
print("\n--- I. Generating Word Clouds ---")
def generate_wc(text_series, filename, color_map, bg):
    full_text = " ".join(text_series.astype(str))
    counts = Counter(full_text.split())
    if not counts: return

    reshaped_counts = {reshaper(k): v for k, v in counts.items()}

    wc = WordCloud(font_path='amiri_font.ttf', width=800, height=400,
                   background_color=bg, colormap=color_map, regexp=r"[\u0600-\u06FF]+").generate_from_frequencies(reshaped_counts)
    wc.to_file(filename)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Word Cloud: {filename.split('_')[-1].split('.')[0].upper()}")
    plt.close()

generate_wc(df_eda[df_eda['label']=='human']['processed_text'], 'reports/figures/eda_wordcloud_human.png', 'viridis', 'white')
generate_wc(df_eda[df_eda['label']=='ai']['processed_text'], 'reports/figures/eda_wordcloud_ai.png', 'Pastel1', 'black')

# ==========================================
# FINAL STEP: ZIP REPORTS
# ==========================================
print("\n" + "="*40)
print("   ZIPPING REPORTS ONLY")
print("="*40)

target_directory = 'reports'
output_zip = 'Step1_Reports_Only_PP_FE.zip'

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    if os.path.exists(target_directory):
        for root, dirs, files in os.walk(target_directory):
            dirs[:] = [d for d in dirs if not d.startswith('.')]
            for file in files:
                if not file.startswith('.'):
                    file_path = os.path.join(root, file)
                    zipf.write(file_path, arcname=file_path)
                    print(f"  Added: {file_path}")

print(f"\nSUCCESS! Download '{output_zip}'")
try:
    from google.colab import files
    files.download(output_zip)
except: pass